In [ ]:
#| default_exp read

## Reading and inspection

Readable notebook views plus implementation-oriented context for co-creation.

The preferred public context API is `context(target, scope=".")`. It covers repository orientation, one-file inspection, one chapter, one cell id, and one concrete implementation without exposing raw notebook JSON.

The focused readers remain small building blocks for the unified function.

Reading is the first problem to solve for notebook automation. An agent should not have to inspect raw `.ipynb` JSON just to learn what a project, notebook, chapter, or implementation contains, and a human reviewer should not have to scroll through outputs and metadata to find the code.

This notebook builds compact context views for four common questions: what project is this, what is in this file, what belongs to this chapter, and what should I know before changing this symbol.

The reader is intentionally a triage tool before it is a renderer. Start with `context(target, scope=".")`; use `target="project"`, a notebook path/name, a chapter title, a cell id, or a Python symbol.

```python
context("project", scope="nbs")
context("nbs/02_write.ipynb")
context("write_nb", scope="nbs/02_write.ipynb")
```

### Production contract

The reading tools are production core. `project_context` must return README context, notebook filenames, and file docstrings; `file_context` must include imports, header docs, Markdown, and definition summaries with regex filters; `chapter_context` must include the notebook head plus one selected section; and `symbol_context` must explain one implementation with nearby docs, examples/tests, callers, and depth-controlled callees.

In [ ]:
from contextlib import redirect_stdout
from io import StringIO
from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import mk_cell, new_nb, read_nb, write_nb
from nbskill.read import chapter_context as _example_chapter_context
from nbskill.read import file_context as _example_file_context
from nbskill.read import project_context as _example_project_context
from nbskill.read import symbol_context as _example_symbol_context
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
with write_demo_notebook("01_read_example.ipynb") as path:
    write_nb(new_nb([
        mk_cell("# Reader\nFile preamble.", cell_type="markdown"),
        mk_cell("preamble = True", cell_type="code"),
        mk_cell("## Demo\nWhy this function exists.", cell_type="markdown"),
        mk_cell("#| export\ndef answer():\n    \"\"\"Return the demo answer.\"\"\"\n    return 42", cell_type="code"),
        mk_cell("assert answer() == 42", cell_type="code"),
        mk_cell("answer()", cell_type="code"),
    ]), path)
    print("project_context")
    _example_project_context(str(path))
    print("\nfile_context")
    _example_file_context(str(path), include_re="answer")
    print("\nchapter_context")
    _example_chapter_context(str(path), name="Demo")
    print("\nsymbol_context")
    _example_symbol_context(str(path), "answer", depth=0)


In [ ]:
#| export
import ast
import copy
import json
import re
import shlex
from pathlib import Path

from fastcore.nbio import read_nb

from nbskill.foundation import (
    cell_matches_type, cell_prefix, cell_source, chapter_index_set,
    cli_return, find_cell_by_id, first_line, is_definition_node,
    is_export_directive, is_exported_code_cell, matches_filter,
    notebook_paths, source_without_directives, with_context,
)

### Output shapes

The context readers serve four attention levels. `project_context` is a repository map, `file_context` is a notebook map with Markdown and definitions, `chapter_context` is an unnumbered chapter view with the notebook head included, and `symbol_context` is the implementation-oriented view with examples, callers, and callees.

In [ ]:
#| export
def _format_overview(items, show_ids=False):
    lines = []
    for idx, cell in items:
        summary = first_line(cell.source)
        lines.append(f"{cell_prefix(idx, cell, show_ids)} | {summary}")
    return "\n".join(lines)

In [ ]:
#| export
def _markdown_overview(cell, include_docs=False):
    source = cell_source(cell).strip()
    headings = []
    for line in source.splitlines():
        text = line.strip()
        if re.match(r"^#{1,6}\s+", text): headings.append(text)
    if headings: return headings
    return source.splitlines() if include_docs and source else []

In [ ]:
#| export
def _definition_lines(node, indent=""):
    tmp = copy.deepcopy(node)
    tmp.body = [ast.Pass()]
    ast.fix_missing_locations(tmp)
    lines = []
    for line in ast.unparse(tmp).splitlines():
        if line.strip() == "pass": continue
        lines.append(f"{indent}{line}" if line else line)
    return lines

In [ ]:
#| export
def _docstring_lines(node, indent="    "):
    doc = ast.get_docstring(node)
    if not doc: return []
    lines = doc.splitlines()
    quote = chr(34) * 3
    if len(lines) == 1: return [f"{indent}{quote}{lines[0]}{quote}"]
    return [indent + quote, *[f"{indent}{line}" for line in lines], indent + quote]

In [ ]:
#| export
def _function_overview(node, indent=""):
    return [*_definition_lines(node, indent=indent), *_docstring_lines(node, indent=indent + "    ")]

In [ ]:
#| export
def _class_overview(node):
    lines = [*_definition_lines(node), *_docstring_lines(node)]
    methods = [child for child in node.body if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef))]
    for method in methods:
        if lines and lines[-1] != "": lines.append("")
        lines += _function_overview(method, indent="    ")
    return lines

In [ ]:
#| export
def _code_overview(cell):
    try: tree = ast.parse(cell.source)
    except SyntaxError: return []
    lines = []
    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)): lines.append(ast.unparse(node))
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): lines += _function_overview(node)
        elif isinstance(node, ast.ClassDef): lines += _class_overview(node)
        if lines and lines[-1] != "": lines.append("")
    if lines and lines[-1] == "": lines.pop()
    return lines

In [ ]:
#| export
def _format_headers(items, include_docs=False):
    chunks = []
    for idx, cell in items:
        if cell.cell_type == "markdown": lines = _markdown_overview(cell, include_docs=include_docs)
        elif cell.cell_type == "code": lines = _code_overview(cell)
        else: lines = []
        if lines: chunks.append(f"{cell_prefix(idx, cell, True)}\n" + "\n".join(lines))
    return "\n\n".join(chunks)

In [ ]:
#| export
def _format_source(source, line_numbers=False):
    if not line_numbers: return source
    lines = source.splitlines() or [""]
    return "\n".join(f"{idx} | {line}" for idx, line in enumerate(lines, start=1))


def _format_full(items, show_ids=False, line_numbers=False):
    chunks = []
    for idx, cell in items:
        chunks.append(f"{cell_prefix(idx, cell, show_ids)}\n{_format_source(cell_source(cell), line_numbers=line_numbers)}")
    return "\n\n".join(chunks)

### A small query language

Automation needs stable selectors, but humans need short commands. The query helpers accept aliases like `id`, `type`, `chapter`, and `contains`, then normalize them into one internal selection shape.

In [ ]:
#| export
_QUERY_KEY_ALIASES = {
    "id": "cell_id",
    "cell": "cell_id",
    "cell_id": "cell_id",
    "chapter": "chapter",
    "type": "cell_type",
    "class": "cell_type",
    "cell_type": "cell_type",
    "contains": "contains",
    "text": "contains",
    "regex": "regex",
    "re": "regex",
}

In [ ]:
#| export
def _normalize_query_key(key):
    name = _QUERY_KEY_ALIASES.get(str(key).strip().lower())
    if name is None:
        choices = ", ".join(sorted(_QUERY_KEY_ALIASES))
        raise ValueError(f"Unknown query key {key!r}; use one of: {choices}")
    return name

In [ ]:
#| export
def _normalize_query_dict(spec):
    return {_normalize_query_key(key): None if value is None else str(value) for key, value in dict(spec).items()}

In [ ]:
#| export
def _parse_query_terms(text):
    spec, bare = {}, []
    for term in shlex.split(str(text)):
        sep = "=" if "=" in term else ":" if ":" in term else None
        if sep is None:
            bare.append(term)
            continue
        key, value = term.split(sep, 1)
        spec[_normalize_query_key(key)] = value
    if bare and "contains" not in spec: spec["contains"] = " ".join(bare)
    return spec

In [ ]:
#| export
def _parse_query(query):
    if query is None: return [{}]
    if isinstance(query, dict): return [_normalize_query_dict(query)]
    if isinstance(query, (list, tuple)):
        specs = []
        for item in query: specs.extend(_parse_query(item))
        return specs or [{}]

    text = str(query).strip()
    if not text: return [{}]
    try: parsed = json.loads(text)
    except json.JSONDecodeError:
        return [_parse_query_terms(part) for part in text.split(";") if part.strip()]
    return _parse_query(parsed)

In [ ]:
#| export
def _select_query_items(nb, spec):
    items = [find_cell_by_id(nb.cells, spec["cell_id"])] if spec.get("cell_id") else list(enumerate(nb.cells))
    if spec.get("chapter") is not None:
        chapter_idxs = chapter_index_set(nb.cells, spec["chapter"])
        items = [(i, c) for i, c in items if i in chapter_idxs]
    if spec.get("cell_type") is not None: items = [(i, c) for i, c in items if cell_matches_type(c, spec["cell_type"])]
    if spec.get("contains") is not None: items = [(i, c) for i, c in items if spec["contains"] in c.source]
    if spec.get("regex") is not None: items = [(i, c) for i, c in items if matches_filter(c.source, spec["regex"])]
    return items

In [ ]:
#| export
def _cell_defined_symbols(cell):
    if getattr(cell, "cell_type", None) != "code": return []
    try: tree = ast.parse(source_without_directives(cell_source(cell)))
    except SyntaxError: return []
    symbols = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): symbols.append(node.name)
        if isinstance(node, ast.ClassDef):
            for child in node.body:
                if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)): symbols.append(f"{node.name}.{child.name}")
    return symbols

In [ ]:
#| export
def _format_usage_for_items(path, items, max_symbols=4):
    symbols = []
    for _, cell in items: symbols.extend(_cell_defined_symbols(cell))
    symbols = list(dict.fromkeys(symbols))
    if not symbols: return ""
    shown = symbols[:max_symbols]
    try:
        from nbskill.graph import symbol_usage_summary
        text = symbol_usage_summary(path, shown)
    except Exception as exc:
        return f"Usage unavailable: {type(exc).__name__}: {exc}"
    if len(symbols) > len(shown):
        text = f"{text}\n... {len(symbols) - len(shown)} more symbols omitted ..."
    return text

In [ ]:
#| export
def _context_result(kind, text, verbose=True, **data):
    result = {"kind": kind, "text": text, **data}
    if verbose and text: print(text)
    return result


def _context_root(path):
    start = Path(path).expanduser()
    if start.suffix == ".ipynb": start = start.parent
    if not start.exists() and start.suffix: start = start.parent
    start = start.resolve() if start.exists() else (Path.cwd() / start).resolve()
    for item in [start, *start.parents]:
        if (item / "README.md").exists(): return item
    return start


def _readme_context(root, max_sections=3):
    readme = Path(root) / "README.md"
    if not readme.exists(): return []
    sections, current = [], []
    for line in readme.read_text(encoding="utf-8", errors="ignore").splitlines():
        if re.match(r"^#{1,2}\s+", line) and current:
            sections.append("\n".join(current).strip())
            current = []
        if line.strip() or current: current.append(line)
    if current: sections.append("\n".join(current).strip())
    return [section for section in sections if section][:max_sections]

In [ ]:
#| export
def _notebook_header_items(cells):
    items, started = [], False
    for idx, cell in enumerate(cells):
        if getattr(cell, "cell_type", None) != "markdown":
            if started: break
            continue
        source = cell_source(cell).strip()
        if re.search(r"^##\s+", source, flags=re.MULTILINE): break
        if re.search(r"^#\s+", source, flags=re.MULTILINE): started = True
        if started: items.append((idx, cell))
    return items


def _notebook_docstring(path):
    nb = read_nb(path)
    items = _notebook_header_items(nb.cells)
    return {
        "path": str(path),
        "cells": [
            {"cell_id": getattr(cell, "id", ""), "source": cell_source(cell).strip()}
            for _, cell in items if cell_source(cell).strip()
        ],
    }


def _context_match(text, include_re=None, exclude_re=None):
    text = str(text or "")
    if include_re and not re.search(include_re, text, flags=re.MULTILINE): return False
    if exclude_re and re.search(exclude_re, text, flags=re.MULTILINE): return False
    return True


def _format_context_blocks(title, blocks):
    lines = [title]
    for heading, body in blocks:
        if not body: continue
        lines.extend(["", heading])
        if isinstance(body, str): lines.append(body)
        else: lines.extend(body)
    return "\n".join(lines).rstrip()


In [ ]:
#| export
def _import_lines(cell):
    if getattr(cell, "cell_type", None) != "code": return []
    try: tree = ast.parse(source_without_directives(cell_source(cell)))
    except SyntaxError: return []
    return [ast.unparse(node) for node in tree.body if isinstance(node, (ast.Import, ast.ImportFrom))]


def _definition_record(path, idx, cell, node, symbol=None, kind=None):
    symbol = symbol or getattr(node, "name", "")
    kind = kind or ("class" if isinstance(node, ast.ClassDef) else "function")
    lines = _class_overview(node) if isinstance(node, ast.ClassDef) else _function_overview(node)
    return {
        "path": str(path),
        "cell_id": getattr(cell, "id", ""),
        "cell_idx": idx,
        "symbol": symbol,
        "kind": kind,
        "text": "\n".join(lines),
    }


def _definition_records(path, nb):
    records = []
    for idx, cell in enumerate(nb.cells):
        if getattr(cell, "cell_type", None) != "code": continue
        try: tree = ast.parse(source_without_directives(cell_source(cell)))
        except SyntaxError: continue
        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                records.append(_definition_record(path, idx, cell, node))
    return records


In [ ]:
#| export
def _output_value_text(value):
    if isinstance(value, list): return "".join(str(item) for item in value)
    return "" if value is None else str(value)


def _output_text(output):
    otype = output.get("output_type")
    if otype == "stream": return _output_value_text(output.get("text", ""))
    if otype == "error": return chr(10).join(output.get("traceback") or [f"{output.get('ename', '')}: {output.get('evalue', '')}"])
    if otype in {"execute_result", "display_data"}:
        data = output.get("data") or {}
        for mime in ("text/plain", "text/markdown", "text/html"):
            value = data.get(mime)
            if value is not None: return _output_value_text(value)
    return ""


def _cell_output_text(cell):
    return "".join(_output_text(output) for output in getattr(cell, "outputs", []) or []).strip()


def _source_for_node(cell, node):
    source = source_without_directives(cell_source(cell))
    segment = ast.get_source_segment(source, node)
    if segment: return segment.strip()
    lines = source.splitlines()
    start = max(getattr(node, "lineno", 1) - 1, 0)
    end = getattr(node, "end_lineno", start + 1)
    return chr(10).join(lines[start:end]).strip()


In [ ]:
#| export
def _symbol_short_name(symbol): return str(symbol).rsplit(".", 1)[-1]


def _name_parts(node):
    if isinstance(node, ast.Name): return [node.id]
    if isinstance(node, ast.Attribute):
        base = _name_parts(node.value)
        return [*base, node.attr] if base else [node.attr]
    return []


def _call_name(node):
    parts = _name_parts(node)
    return ".".join(parts) if parts else None


def _call_matches_context_symbol(name, symbol):
    name = str(name or "")
    short = _symbol_short_name(symbol)
    return name == symbol or name.rsplit(".", 1)[-1] == short


def _cell_calls_symbol(cell, symbol):
    if getattr(cell, "cell_type", None) != "code": return False
    try: tree = ast.parse(source_without_directives(cell_source(cell)))
    except SyntaxError: return False
    for node in ast.walk(tree):
        if isinstance(node, ast.Call) and _call_matches_context_symbol(_call_name(node.func), symbol): return True
    return False


In [ ]:
#| export
def _markdown_mentions(nb, symbol):
    short = _symbol_short_name(symbol)
    items = []
    for idx, cell in enumerate(nb.cells):
        if getattr(cell, "cell_type", None) != "markdown": continue
        source = cell_source(cell)
        if symbol in source or re.search(rf"\b{re.escape(short)}\b", source): items.append((idx, cell))
    return items


def _example_test_records(nb, symbol):
    records = []
    for idx, cell in enumerate(nb.cells):
        if not _cell_calls_symbol(cell, symbol): continue
        kind = "test" if cell_matches_type(cell, "test") else "example" if cell_matches_type(cell, "example") else "usage"
        records.append({
            "cell_id": getattr(cell, "id", ""),
            "cell_idx": idx,
            "kind": kind,
            "source": cell_source(cell).strip(),
            "output": _cell_output_text(cell),
        })
    return records


def _symbol_location(path, nb, symbol):
    idx = _find_symbol_cell(nb, symbol)
    cell = nb.cells[idx]
    node = _find_symbol_node(cell, symbol)
    return idx, cell, node


In [ ]:
#| export
def _callee_summary(path, symbol, depth, seen=None):
    if depth <= 0: return []
    seen = set() if seen is None else seen
    if symbol in seen: return []
    seen.add(symbol)
    try:
        from nbskill.graph import symbol_graph_data
        data = symbol_graph_data(path, symbol)
    except Exception:
        return []
    items = []
    for callee in data.get("callees", []):
        if callee in seen: continue
        definitions = data.get("graph", {}).get("definitions", [])
        definition = next((record for record in definitions if record.get("symbol") == callee), None)
        summary = ""
        if definition:
            try:
                callee_nb = read_nb(definition["path"])
                callee_cell = callee_nb.cells[definition["cell_idx"]]
                summary = "\n".join(_symbol_signature_text(callee_cell, callee))
            except Exception:
                summary = ""
        item = {
            "symbol": callee,
            "path": definition.get("path") if definition else "",
            "cell_id": definition.get("cell_id") if definition else "",
            "summary": summary,
            "callees": _callee_summary(path, callee, depth - 1, seen),
        }
        items.append(item)
    return items


def _format_callee_items(items, indent=""):
    lines = []
    for item in items:
        loc = f" {item['path']} id={item['cell_id']}" if item.get("path") else ""
        lines.append(f"{indent}- {item['symbol']}:{loc}".rstrip())
        if item.get("summary"):
            lines.extend(f"{indent}  {line}" for line in item["summary"].splitlines())
        lines.extend(_format_callee_items(item.get("callees", []), indent=indent + "  "))
    return lines


### The public notebook reader
The preferred public reader is `context(target, scope=".")`. `target` can be `project`, a notebook path, a cell id, a chapter title, or a Python symbol; `scope` only narrows where notebook targets are searched.
The older focused readers remain as small internal building blocks, but the simple path is one target plus one optional search scope.

In [ ]:
#| export
def _chapter_title_from_cell(cell):
    if getattr(cell, "cell_type", None) != "markdown": return None
    for line in cell_source(cell).splitlines():
        match = re.match(r"^#{1,6}\s+(.+?)\s*$", line.strip())
        if match: return match.group(1).strip()
    return None

In [ ]:
#| export
def _normalize_chapter_title(value):
    return re.sub(r"\s+", " ", str(value).strip().lower())

In [ ]:
#| export
def _chapter_spans_for_nb(cells):
    starts = [(idx, title) for idx, cell in enumerate(cells) if (title := _chapter_title_from_cell(cell))]
    if not starts: return [dict(title="Notebook", start=0, end=len(cells))]
    spans = []
    for pos, (start, title) in enumerate(starts):
        end = starts[pos + 1][0] if pos + 1 < len(starts) else len(cells)
        spans.append(dict(title=title, start=start, end=end))
    return spans

In [ ]:
#| export
def _notebook_head_items(cells):
    spans = _chapter_spans_for_nb(cells)
    head_end = spans[0]["start"] if spans else len(cells)
    return [(idx, cells[idx]) for idx in range(head_end)]

In [ ]:
#| export
def _chapter_span_for_index(cells, idx):
    spans = _chapter_spans_for_nb(cells)
    if spans and idx < spans[0]["start"]:
        return dict(title="Notebook head", start=0, end=spans[0]["start"])
    for span in spans:
        if span["start"] <= idx < span["end"]: return span
    raise ValueError(f"Cell index {idx} is outside the notebook")

In [ ]:
#| export
def _chapter_matches(span, name):
    title = span["title"]
    if matches_filter(title, name): return True
    wanted, candidate = _normalize_chapter_title(name), _normalize_chapter_title(title)
    return bool(wanted and (wanted in candidate or candidate in wanted))

In [ ]:
#| export
def _one_chapter_span(cells, name):
    spans = _chapter_spans_for_nb(cells)
    matches = [span for span in spans if _chapter_matches(span, name)]
    if len(matches) == 1: return matches[0]
    titles = ", ".join(span["title"] for span in spans[:8]) or "none"
    if not matches: raise ValueError(f"No chapter matches {name!r}. Available chapters: {titles}")
    matched = ", ".join(span["title"] for span in matches)
    raise ValueError(f"Chapter {name!r} matches multiple chapters: {matched}")

In [ ]:
#| export
def _query_items(nb, query):
    items, seen = [], set()
    for spec in _parse_query(query):
        for idx, cell in _select_query_items(nb, spec):
            if idx in seen: continue
            seen.add(idx)
            items.append((idx, cell))
    return items

In [ ]:
#| export
def _selected_chapter_span(nb, query=None, name=None, any_cell_id=None):
    selectors = [value is not None for value in (query, name, any_cell_id)]
    if sum(selectors) != 1: raise ValueError("Pass exactly one of query, name, or any_cell_id")
    if name is not None: return _one_chapter_span(nb.cells, name)
    if any_cell_id is not None:
        idx, _ = find_cell_by_id(nb.cells, any_cell_id)
        return _chapter_span_for_index(nb.cells, idx)
    items = _query_items(nb, query)
    if not items: raise ValueError(f"No cells match query {query!r}")
    return _chapter_span_for_index(nb.cells, items[0][0])

In [ ]:
#| export
def _chapter_items(nb, span):
    idxs = set()
    items = []
    for idx, cell in [*_notebook_head_items(nb.cells), *[(i, nb.cells[i]) for i in range(span["start"], span["end"])]]:
        if idx in idxs: continue
        idxs.add(idx)
        items.append((idx, cell))
    return items

In [ ]:
#| export
def _selected_cell_item(nb, query=None, id=None):
    if (query is None) == (id is None): raise ValueError("Pass exactly one of query or id")
    if id is not None: return find_cell_by_id(nb.cells, id)
    items = _query_items(nb, query)
    if len(items) == 1: return items[0]
    if not items: raise ValueError(f"No cells match query {query!r}")
    preview = _format_overview(items[:12], show_ids=True)
    raise ValueError(f"Query {query!r} matched {len(items)} cells; narrow it or pass id.\n{preview}")

In [ ]:
#| export
def project_context(
    path: str = ".",  # Project directory, notebook path, or notebook glob root
    verbose: bool = True,  # Print rendered context; pass False to only return structured data
):
    "Show README sections, notebook filenames, and notebook file docstrings."
    root = _context_root(path)
    notebooks = [str(item) for item in notebook_paths(path)]
    docstrings = [_notebook_docstring(item) for item in notebooks]
    readme_sections = _readme_context(root)
    nl = chr(10)
    blocks = [
        ("Notebooks", [f"- {item}" for item in notebooks]),
        ("Notebook Docstrings", [
            f"{item['path']}{nl}" + (nl * 2).join(cell["source"] for cell in item["cells"])
            for item in docstrings if item["cells"]
        ]),
        ("README", (nl * 2).join(readme_sections)),
    ]
    text = _format_context_blocks(f"Project context: {root}{nl}Requested path: {path}", blocks)
    return _context_result(
        "project_context", text, verbose=verbose, root=str(root), requested_path=str(path),
        readme_sections=readme_sections, notebooks=notebooks, docstrings=docstrings,
    )

In [ ]:
#| export
def chapter_context(
    path: str,  # Notebook path
    query: str | None = None,  # Query for any cell inside the chapter
    name: str | None = None,  # Chapter title string or regex
    any_cell_id: str | None = None,  # Any cell id inside the chapter
    verbose: bool = True,  # Print rendered context; pass False to only return structured data
):
    "Show the notebook head plus one selected chapter."
    nb = read_nb(path)
    span = _selected_chapter_span(nb, query=query, name=name, any_cell_id=any_cell_id)
    items = _chapter_items(nb, span)
    text = _format_full(items, show_ids=True, line_numbers=False)
    cells = [{"cell_id": getattr(cell, "id", ""), "cell_idx": idx, "cell_type": cell.cell_type, "source": cell_source(cell)} for idx, cell in items]
    return _context_result(
        "chapter_context", text, verbose=verbose, path=str(path), query=query,
        name=name, any_cell_id=any_cell_id, chapter=span, cells=cells,
    )


In [ ]:
#| export
def file_context(
    path: str,  # Notebook path
    include_re: str | None = None,  # Optional regex for markdown and definition items to include
    exclude_re: str | None = None,  # Optional regex for markdown and definition items to exclude
    verbose: bool = True,  # Print rendered context; pass False to only return structured data
):
    "Show imports, header docs, Markdown, and definition summaries for one notebook."
    nb = read_nb(path)
    imports = []
    for _, cell in enumerate(nb.cells): imports.extend(_import_lines(cell))
    imports = list(dict.fromkeys(imports))
    header = [{"cell_id": getattr(cell, "id", ""), "cell_idx": idx, "source": cell_source(cell).strip()} for idx, cell in _notebook_header_items(nb.cells)]
    markdown = [
        {"cell_id": getattr(cell, "id", ""), "cell_idx": idx, "source": cell_source(cell).strip()}
        for idx, cell in enumerate(nb.cells)
        if getattr(cell, "cell_type", None) == "markdown" and _context_match(cell_source(cell), include_re, exclude_re)
    ]
    definitions = [item for item in _definition_records(path, nb) if _context_match(f"{item['symbol']}\n{item['text']}", include_re, exclude_re)]
    blocks = [
        ("Header", [item["source"] for item in header]),
        ("Imports", imports),
        ("Markdown", [f"Cell id={item['cell_id']}\n{item['source']}" for item in markdown]),
        ("Definitions", [f"Cell id={item['cell_id']} {item['kind']} {item['symbol']}\n{item['text']}" for item in definitions]),
    ]
    text = _format_context_blocks(f"File context: {path}", blocks)
    return _context_result(
        "file_context", text, verbose=verbose, path=str(path), include_re=include_re,
        exclude_re=exclude_re, imports=imports, header=header, markdown=markdown, definitions=definitions,
    )


In [ ]:
#| export
def _annotation_name(annotation):
    if annotation is None: return None
    if isinstance(annotation, ast.Name): return annotation.id
    if isinstance(annotation, ast.Attribute): return annotation.attr
    if isinstance(annotation, ast.Constant): return annotation.value
    return ast.unparse(annotation)

In [ ]:
#| export
def _first_arg_annotation(node):
    args = node.args.posonlyargs or node.args.args
    return _annotation_name(args[0].annotation) if args else None

In [ ]:
#| export
def _node_defines_symbol(node, symbol):
    parts = symbol.split(".")
    name = parts[-1]
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == name:
        return True
    if len(parts) < 2: return False

    cls_name, meth_name = parts[-2], parts[-1]
    if isinstance(node, ast.ClassDef) and node.name == cls_name:
        return any(isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name == meth_name for child in node.body)
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == meth_name:
        return _first_arg_annotation(node) == cls_name
    return False

In [ ]:
#| export
def _cell_defines_symbol(cell, symbol):
    if cell.cell_type != "code": return False
    try: tree = ast.parse(source_without_directives(cell.source))
    except SyntaxError: return False
    return any(_node_defines_symbol(node, symbol) for node in tree.body)

In [ ]:
#| export
def _find_symbol_cell(nb, symbol):
    for idx, cell in enumerate(nb.cells):
        if _cell_defines_symbol(cell, symbol): return idx
    raise ValueError(f"Could not find symbol {symbol!r}")

In [ ]:
#| export
def _find_symbol_node(cell, symbol):
    if getattr(cell, "cell_type", None) != "code": return None
    try: tree = ast.parse(source_without_directives(cell.source))
    except SyntaxError: return None
    parts = symbol.split(".")
    for node in tree.body:
        if len(parts) == 1 and isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and node.name == symbol:
            return node
        if _node_defines_symbol(node, symbol):
            if isinstance(node, ast.ClassDef) and len(parts) > 1:
                name = parts[-1]
                return next((child for child in node.body if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)) and child.name == name), node)
            return node
    return None

In [ ]:
#| export
def _previous_markdown(cells, idx, limit):
    docs = []
    pos = idx - 1
    while pos >= 0 and len(docs) < limit:
        cell = cells[pos]
        if getattr(cell, "cell_type", None) != "markdown": break
        docs.append((pos, cell))
        pos -= 1
    return list(reversed(docs))

In [ ]:
#| export
def _following_examples(cells, idx, limit):
    examples = []
    pos = idx + 1
    while pos < len(cells) and len(examples) < limit:
        cell = cells[pos]
        if is_exported_code_cell(cell): break
        if getattr(cell, "cell_type", None) in {"markdown", "code"}: examples.append((pos, cell))
        pos += 1
    return examples

In [ ]:
#| export
def _symbol_signature_text(cell, symbol):
    node = _find_symbol_node(cell, symbol)
    if node is None: return _code_overview(cell)
    if isinstance(node, ast.ClassDef): return _class_overview(node)
    return _function_overview(node)

In [ ]:
#| export
def _usage_group_locations(raw_locations):
    raw = str(raw_locations or "").strip()
    if not raw or raw == "none": return [], 0
    grouped = {}
    for item in [part.strip() for part in raw.split(";") if part.strip()]:
        path, _, cell_id = item.rpartition(" id=")
        if not path: path, cell_id = item, ""
        grouped.setdefault(path, []).append(cell_id)
    return list(grouped.items()), sum(len(ids) for ids in grouped.values())

In [ ]:
#| export
def _format_usage_locations(label, raw_locations, max_paths=4, max_ids=4):
    groups, total = _usage_group_locations(raw_locations)
    if not total: return [f"{label}: none"]
    cell_word = "cell" if total == 1 else "cells"
    notebook_word = "notebook" if len(groups) == 1 else "notebooks"
    lines = [f"{label}: {total} {cell_word} across {len(groups)} {notebook_word}"]
    for path, ids in groups[:max_paths]:
        shown_ids = [item for item in ids[:max_ids] if item]
        suffix = f": {', '.join(shown_ids)}" if shown_ids else ""
        if len(ids) > max_ids: suffix += f", +{len(ids) - max_ids} more"
        lines.append(f"- {path}{suffix}")
    if len(groups) > max_paths: lines.append(f"- +{len(groups) - max_paths} more notebooks")
    return lines

In [ ]:
#| export
def _raw_caller_usage_lines(raw_lines):
    if "Caller usages:" not in raw_lines: return []
    start = raw_lines.index("Caller usages:") + 1
    return [line for line in raw_lines[start:] if line.startswith("- ")]

In [ ]:
#| export
def _format_symbol_usage(path, symbol):
    try:
        from nbskill.graph import symbol_usage_summary
        raw = symbol_usage_summary(path, [symbol])
    except Exception as exc:
        return [f"Usage unavailable: {type(exc).__name__}: {exc}"]
    if not raw: return []
    raw_lines = raw.splitlines()
    line = raw_lines[0]
    prefix = f"{symbol}: callers="
    if not line.startswith(prefix) or "; callees=" not in line: return raw_lines
    callers, _, callees = line[len(prefix):].partition("; callees=")
    lines = _format_usage_locations("Callers", callers)
    caller_usage = _raw_caller_usage_lines(raw_lines)
    if caller_usage:
        lines.append("Caller usages:")
        lines.extend(caller_usage)
    callee_items = [item.strip() for item in callees.split(";") if item.strip() and item.strip() != "none"]
    lines.append(f"Callees: {', '.join(callee_items)}" if callee_items else "Callees: none")
    return lines

In [ ]:
#| export
def _format_symbol_doc(path, nb, symbol, context=2, source=False, show_ids=False):
    idx = _find_symbol_cell(nb, symbol)
    cell = nb.cells[idx]
    lines = [f"Symbol {symbol}", f"Location: {path} {cell_prefix(idx, cell, show_ids)}"]
    docs = _previous_markdown(nb.cells, idx, context)
    if docs:
        lines.append("")
        lines.append("Docs")
        for doc_idx, doc_cell in docs:
            if show_ids: lines.append(cell_prefix(doc_idx, doc_cell, show_ids))
            lines.append(doc_cell.source.strip())
    signature = _symbol_signature_text(cell, symbol)
    if signature:
        lines.append("")
        lines.append("Definition")
        lines.extend(signature)
    if source:
        lines.append("")
        lines.append("Source cell")
        lines.append(cell.source.strip())
    examples = _following_examples(nb.cells, idx, context)
    if examples:
        lines.append("")
        lines.append("Examples/tests")
        for ex_idx, ex_cell in examples:
            lines.append(cell_prefix(ex_idx, ex_cell, show_ids))
            lines.append(ex_cell.source.strip())
    usage = _format_symbol_usage(path, symbol)
    if usage:
        lines.append("")
        lines.append("Usage")
        lines.extend(usage)
    return "\n".join(lines)

In [ ]:
#| export
def symbol_context(
    path: str,  # Notebook path
    symbol: str,  # Function, class, or Class.method to inspect
    depth: int = 1,  # Callee summary depth; 0 keeps only direct implementation context
    verbose: bool = True,  # Print rendered context; pass False to only return structured data
):
    "Show implementation, local docs/examples/tests, callers, and callees for a notebook symbol."
    nb = read_nb(path)
    idx, cell, node = _symbol_location(path, nb, symbol)
    source = _source_for_node(cell, node) if node is not None else cell_source(cell).strip()
    markdown = [{"cell_id": getattr(item, "id", ""), "cell_idx": pos, "source": cell_source(item).strip()} for pos, item in _markdown_mentions(nb, symbol)]
    examples = _example_test_records(nb, symbol)
    callers, callees = [], []
    if depth > 0:
        try:
            from nbskill.graph import symbol_graph_data
            graph_data = symbol_graph_data(path, symbol)
            callers = graph_data.get("caller_usages", [])
        except Exception:
            callers = []
        callees = _callee_summary(path, symbol, depth)
    nl = chr(10)
    blocks = [
        ("Implementation", source),
        ("Examples/tests", [
            f"Cell id={item['cell_id']} {item['kind']}{nl}{item['source']}" + (f"{nl}Output:{nl}{item['output']}" if item.get("output") else "")
            for item in examples
        ]),
        ("Callers", [
            f"- {item.get('path')} id={item.get('cell_id')} line {item.get('lineno')}: {item.get('line')}"
            for item in callers
        ]),
        (f"Callees depth {depth}", _format_callee_items(callees)),
        ("Markdown mentions", [f"Cell id={item['cell_id']}{nl}{item['source']}" for item in markdown]),
    ]
    text = _format_context_blocks(f"Symbol context: {symbol}{nl}Location: {path} {cell_prefix(idx, cell, True)}", blocks)
    return _context_result(
        "symbol_context", text, verbose=verbose, path=str(path), symbol=symbol,
        depth=depth, location={"cell_id": getattr(cell, "id", ""), "cell_idx": idx},
        source=source, markdown=markdown, examples=examples, callers=callers, callees=callees,
    )

In [ ]:
with write_demo_notebook("01_read_doc.ipynb") as path:
    nb = new_nb([
        mk_cell("# Demo project\nFile-level note.", cell_type="markdown"),
        mk_cell("## Addition\nThis explains add.", cell_type="markdown"),
        mk_cell("#| export\ndef add(a, b):\n    \"\"\"Add two values.\"\"\"\n    return a + b", cell_type="code"),
        mk_cell("assert add(1, 2) == 3", cell_type="code"),
    ])
    write_nb(nb, path)
    result = symbol_context(str(path), "add", depth=0, verbose=False)
    assert "Add two values." in result["text"]
    assert "This explains add." in result["text"]
    assert "assert add(1, 2) == 3" in result["text"]


In [ ]:
def _write_read_sample_notebook(path):
    example = mk_cell("add(2, 3)", cell_type="code")
    example.outputs = [{
        "output_type": "execute_result",
        "execution_count": 1,
        "metadata": {},
        "data": {"text/plain": "5"},
    }]
    nb = new_nb([
        mk_cell("# Sample tool\nNotebook-level note.", cell_type="markdown"),
        mk_cell("More file-level context.", cell_type="markdown"),
        mk_cell("import math", cell_type="code"),
        mk_cell("## Math\nTiny rationale for add.", cell_type="markdown"),
        mk_cell(
            "#| export\ndef double(value):\n"
            "    \"\"\"Double a value.\"\"\"\n"
            "    return value * 2\n\n"
            "def add(a, b):\n"
            "    \"\"\"Add values.\"\"\"\n"
            "    return double(a) + b\n\n"
            "class Calculator:\n"
            "    \"\"\"Calculate values.\"\"\"\n"
            "    def __init__(self, base):\n"
            "        \"\"\"Store the base value.\"\"\"\n"
            "        self.base = base\n\n"
            "    def total(self, value):\n"
            "        \"\"\"Add value to the base.\"\"\"\n"
            "        return add(self.base, value)",
            cell_type="code",
        ),
        mk_cell("assert add(1, 2) == 4", cell_type="code"),
        example,
        mk_cell("### Detail\nNested rationale mentioning add.", cell_type="markdown"),
        mk_cell("detail_value = add(2, 4)", cell_type="code"),
    ])
    write_nb(nb, path)
    return nb


In [ ]:
#| export
def _markdown_heading_level(source):
    for line in str(source or "").splitlines():
        match = re.match(r"^(#{1,6})\s+", line.strip())
        if match: return len(match.group(1))
    return None
def _notebook_context(
    path: str,
    scope: str = ".",
    verbose: bool = True,
    readme_sections: int = 2,
):
    "Show README context, top-level notebook Markdown, imports, and definitions."
    nb = read_nb(path)
    root = _context_root(scope if scope not in (None, "") else path)
    readme = _readme_context(root, max_sections=readme_sections)
    imports = []
    for _, cell in enumerate(nb.cells): imports.extend(_import_lines(cell))
    imports = list(dict.fromkeys(imports))
    head_idxs = {idx for idx, _ in _notebook_header_items(nb.cells)}
    markdown = []
    for idx, cell in enumerate(nb.cells):
        if getattr(cell, "cell_type", None) != "markdown": continue
        source = cell_source(cell).strip()
        level = _markdown_heading_level(source)
        if idx in head_idxs or (level is not None and level <= 2):
            markdown.append({"cell_id": getattr(cell, "id", ""), "cell_idx": idx, "source": source})
    definitions = _definition_records(path, nb)
    blocks = [
        ("README", "\n\n".join(readme)),
        ("Top-level Markdown", [f"Cell id={item['cell_id']}\n{item['source']}" for item in markdown]),
        ("Imports", imports),
        ("Definitions", [f"Cell id={item['cell_id']} {item['kind']} {item['symbol']}\n{item['text']}" for item in definitions]),
    ]
    nl = chr(10)
    title = f"Notebook context: {Path(path).name}{nl}Path: {path}{nl}Project: {root}"
    text = _format_context_blocks(title, blocks)
    return _context_result(
        "notebook_context", text, verbose=verbose, path=str(path), scope=str(scope),
        root=str(root), readme_sections=readme, imports=imports, markdown=markdown, definitions=definitions,
    )

In [ ]:
#| export
def _context_notebooks(scope):
    notebooks = [str(item) for item in notebook_paths(scope or ".")]
    if not notebooks: raise ValueError(f"No notebooks found in scope {scope!r}")
    return notebooks
def _context_existing_notebook(target):
    path = Path(str(target)).expanduser()
    if path.suffix == ".ipynb" and path.exists(): return str(path)
    return None
def _context_named_notebook(target, notebooks):
    wanted = str(target)
    matches = []
    for path in notebooks:
        nb_path = Path(path)
        if wanted in {str(nb_path), nb_path.name, nb_path.stem}: matches.append(path)
    if len(matches) > 1:
        shown = "\n".join(f"- {item}" for item in matches[:8])
        raise ValueError(f"Notebook target {target!r} matched multiple notebooks:\n{shown}")
    return matches[0] if matches else None

In [ ]:
#| export
def _context_cell_matches(target, notebooks):
    matches = []
    for path in notebooks:
        nb = read_nb(path)
        for idx, cell in enumerate(nb.cells):
            if getattr(cell, "id", None) == target:
                matches.append({"path": path, "cell_idx": idx, "cell_id": getattr(cell, "id", "")})
    return matches
def _context_symbol_matches(target, notebooks):
    matches = []
    for path in notebooks:
        nb = read_nb(path)
        try: idx = _find_symbol_cell(nb, target)
        except ValueError: continue
        matches.append({"path": path, "cell_idx": idx, "symbol": str(target)})
    return matches
def _context_chapter_matches(target, notebooks):
    matches = []
    for path in notebooks:
        nb = read_nb(path)
        for span in _chapter_spans_for_nb(nb.cells):
            if _chapter_matches(span, target): matches.append({"path": path, "chapter": span})
    return matches
def _context_one_match(kind, target, matches):
    if len(matches) == 1: return matches[0]
    if not matches: return None
    shown = []
    for item in matches[:8]:
        detail = item.get("cell_id") or item.get("symbol") or item.get("chapter", {}).get("title", "")
        shown.append(f"- {kind}: {item['path']} {detail}".rstrip())
    raise ValueError(f"Target {target!r} matched multiple {kind} contexts:\n" + "\n".join(shown))

In [ ]:
#| export
def _context_symbol_graphs(path, symbols):
    symbols = [str(item) for item in dict.fromkeys(symbols or []) if item]
    if not symbols: return []
    try:
        from nbskill.graph import symbol_graph_data, symbol_graph_public_data
    except Exception:
        return []
    graphs = []
    for symbol in symbols:
        try: graphs.append(symbol_graph_public_data(symbol_graph_data(path, symbol)))
        except Exception: pass
    return graphs
def _context_graph_text(graphs):
    if not graphs: return ""
    lines = ["Symbol graph"]
    for graph in graphs:
        symbol = graph.get("symbol", "")
        callers = graph.get("caller_usages") or graph.get("callers") or []
        callees = graph.get("callees") or []
        lines.append(f"- {symbol}: definitions={len(graph.get('definitions', []))}, callers={len(callers)}, callees={len(callees)}")
        for usage in callers[:5]:
            line = usage.get("line", "")
            loc = f"{usage.get('path')} id={usage.get('cell_id')}"
            lineno = f" line {usage.get('lineno')}" if usage.get("lineno") else ""
            lines.append(f"  caller: {loc}{lineno}: {line}".rstrip())
        for callee in callees[:5]:
            callee_symbol = callee.get("symbol", callee) if isinstance(callee, dict) else callee
            lines.append(f"  callee: {callee_symbol}")
    return "\n".join(lines)
def _context_cell_symbols(path, cell_idx):
    nb = read_nb(path)
    return [item["symbol"] for item in _definition_records(path, nb) if item.get("cell_idx") == cell_idx]
def _context_with_graphs(result, graphs):
    if not graphs: return result
    text = result.get("text", "")
    graph_text = _context_graph_text(graphs)
    return {**result, "text": f"{text}\n\n{graph_text}".rstrip(), "symbol_graphs": graphs}
def _context_as_single(result, target, scope, resolved_kind, verbose=True, **extra):
    return _context_result(
        "context", result["text"], verbose=verbose, target=str(target), scope=str(scope),
        resolved_kind=resolved_kind, selection=result, **extra,
    )
def context(
    target: str = "project",  # project, notebook path/name, chapter title, cell id, or Python symbol
    scope: str = ".",  # Project, folder, glob, or notebook used to narrow target lookup
    verbose: bool = True,  # Print rendered context; pass False to only return structured data
):
    "Return the best notebook-aware context for one target."
    target = "project" if target in (None, "") else str(target)
    scope = "." if scope in (None, "") else str(scope)
    if target == "project":
        result = project_context(scope, verbose=False)
        return _context_as_single(result, target, scope, "project", verbose=verbose)
    path = _context_existing_notebook(target)
    notebooks = _context_notebooks(scope)
    path = path or _context_named_notebook(target, notebooks)
    if path is not None:
        result = _notebook_context(path, scope=scope, verbose=False)
        return _context_as_single(result, target, scope, "notebook", verbose=verbose)
    cell = _context_one_match("cell", target, _context_cell_matches(target, notebooks))
    if cell is not None:
        result = chapter_context(cell["path"], any_cell_id=target, verbose=False)
        graphs = _context_symbol_graphs(cell["path"], _context_cell_symbols(cell["path"], cell["cell_idx"])) if verbose else []
        result = _context_with_graphs(result, graphs)
        return _context_as_single(result, target, scope, "cell", verbose=verbose, symbol_graphs=graphs)
    symbol = _context_one_match("symbol", target, _context_symbol_matches(target, notebooks))
    if symbol is not None:
        result = symbol_context(symbol["path"], target, verbose=False)
        graphs = _context_symbol_graphs(symbol["path"], [target]) if verbose else []
        result = _context_with_graphs(result, graphs)
        return _context_as_single(result, target, scope, "symbol", verbose=verbose, symbol_graphs=graphs)
    chapter = _context_one_match("chapter", target, _context_chapter_matches(target, notebooks))
    if chapter is not None:
        result = chapter_context(chapter["path"], name=chapter["chapter"]["title"], verbose=False)
        return _context_as_single(result, target, scope, "chapter", verbose=verbose)
    raise ValueError(f"Could not resolve context target {target!r} inside scope {scope!r}")

`context` keeps the common call small. For example, `context("add", scope="nbs/01_read.ipynb")` resolves the symbol, prints its implementation, nearby Markdown, examples, callers, and callees. A notebook target such as `context("nbs/01_read.ipynb")` prints README context plus that notebook's top-level Markdown, imports, and definitions.

In [ ]:
#| eval: false
context("add", scope="nbs/01_read.ipynb")

In [ ]:
#| hide
with write_demo_notebook("01_read_single_context.ipynb") as path:
    nb = _write_read_sample_notebook(path)
    symbol = context("add", scope=str(path), verbose=False)
    assert symbol["kind"] == "context"
    assert symbol["resolved_kind"] == "symbol"
    assert "def add(a, b):" in symbol["text"]
    assert "Tiny rationale" in symbol["text"]
    symbol_verbose = context("add", scope=str(path), verbose=True)
    assert symbol_verbose["symbol_graphs"]
    assert "Symbol graph" in symbol_verbose["text"]
    notebook = context(str(path), verbose=False)
    assert notebook["resolved_kind"] == "notebook"
    assert "Notebook context:" in notebook["text"]
    assert "Top-level Markdown" in notebook["text"]
    assert "Nested rationale" not in notebook["text"]
    chapter = context("Math", scope=str(path), verbose=False)
    assert chapter["resolved_kind"] == "chapter"
    assert "assert add(1, 2) == 4" in chapter["text"]
    cell = context(nb.cells[4].id, scope=str(path), verbose=False)
    assert cell["resolved_kind"] == "cell"
    assert "Tiny rationale" in cell["text"]
    cell_verbose = context(nb.cells[4].id, scope=str(path), verbose=True)
    assert cell_verbose["symbol_graphs"]
    project = context("project", scope=str(path), verbose=False)
    assert project["resolved_kind"] == "project"
    assert any(item.endswith("01_read_single_context.ipynb") for item in project["selection"]["notebooks"])

In [ ]:
with write_demo_notebook("01_read_doc.ipynb") as path:
    nb = new_nb([
        mk_cell("# Demo project\nFile-level note.", cell_type="markdown"),
        mk_cell("## Addition\nThis explains add.", cell_type="markdown"),
        mk_cell("#| export\ndef add(a, b):\n    \"\"\"Add two values.\"\"\"\n    return a + b", cell_type="code"),
        mk_cell("assert add(1, 2) == 3", cell_type="code"),
    ])
    write_nb(nb, path)
    out = StringIO()
    with redirect_stdout(out):
        result = symbol_context(str(path), "add", depth=0)
    text = out.getvalue()
    assert result["kind"] == "symbol_context"
    assert "Symbol context: add" in text
    assert "def add(a, b):" in text
    assert "This explains add." in text
    assert "assert add(1, 2) == 3" in text
    assert "Callers" not in text


In [ ]:
with write_demo_notebook("01_read_sample.ipynb") as path:
    _write_read_sample_notebook(path)
    out = StringIO()
    with redirect_stdout(out):
        result = file_context(str(path))
    text = out.getvalue()
    assert result["kind"] == "file_context"
    assert "# Sample tool" in text
    assert "Notebook-level note." in text
    assert "import math" in text
    assert "def add(a, b):" in text
    assert "Add values." in text
    assert "class Calculator:" in text
    assert "    def total(self, value):" in text
    assert "Add value to the base." in text

    filtered = file_context(str(path), include_re="Calculator", verbose=False)
    assert "Calculator" in filtered["text"]
    assert "def add(a, b):" not in filtered["text"]

    excluded = file_context(str(path), exclude_re="Calculator", verbose=False)
    assert "def add(a, b):" in excluded["text"]
    assert "class Calculator:" not in excluded["text"]


In [ ]:
with write_demo_notebook("01_read_sample.ipynb") as path:
    _write_read_sample_notebook(path)
    out = StringIO()
    with redirect_stdout(out):
        result = chapter_context(str(path), name="Math")
    text = out.getvalue()
    assert result["kind"] == "chapter_context"
    assert "Tiny rationale" in text
    assert "assert add(1, 2) == 4" in text
    assert "1 |" not in text

    out = StringIO()
    with redirect_stdout(out):
        chapter_context(str(path), name="Detail")
    text = out.getvalue()
    assert "Nested rationale" in text
    assert "detail_value = add(2, 4)" in text


In [ ]:
with write_demo_notebook("01_read_sample.ipynb") as path:
    _write_read_sample_notebook(path)
    project = project_context(str(path), verbose=False)
    assert project["kind"] == "project_context"
    assert any(item.endswith("01_read_sample.ipynb") for item in project["notebooks"])
    assert "Sample tool" in project["text"]

    shallow = symbol_context(str(path), "add", depth=0, verbose=False)
    assert "def add(a, b):" in shallow["text"]
    assert "Callers" not in shallow["text"]
    assert "Callees depth" not in shallow["text"]

    deep = symbol_context(str(path), "add", depth=1, verbose=False)
    assert "double" in deep["text"]
    assert "Output:" in deep["text"]
    assert "5" in deep["text"]


### Symbol documentation

`symbol_context` answers the symbol-first question: "what should I know before changing this implementation?" It finds the defining function, class, or method, then collects mentioning prose, examples/tests, callers, and optional callee summaries.